#### Name:
#### Date:

For the following exercises, we'll use the packages below:

In [ ]:
# Run this to install the mapping package we'll use
!pip install basemap

In [ ]:
# You're familiar with these first two!
import pandas as pd
import matplotlib.pyplot as plt

# This will allow us to obtain data from elsewhere (section I)
import requests

# This works with pyplot to make plots of maps (section II)
from mpl_toolkits.basemap import Basemap

# I. Querying for data in Python

Scientists often make use of datasets that are collected, synthesized, and maintained by someone else. For example, maybe we want to look at all of the earthquakes that occurred in the continental US today. To do so, we might consider visiting the USGS website and finding a place where we could filter recent earthquakes and download a table with such data.

Python makes this much simpler!

With the *requests* package, we can give python the link to an online database or API. We then specify certain parameters (i.e. the desired date range for our dataset, the location bounds, etc.) and python returns the corresponding data. Below I'll give an example of using the USGS catalog to get all earthquakes within the past day with a magnitude greater than 4.5

In [ ]:
# Our Parameters!
starttime = "2023-04-07" # start of date range
endtime = "2023-04-08" # end of date range
minmag = 4.5 # minimum magnitude

# Building our url - the format is specified by the API
# to query the USGS API further, just copy the below chunk of code!
url = (
    "https://earthquake.usgs.gov/fdsnws/event/1/query?" # Base API endpoint
    f"format=geojson&starttime={starttime}&endtime={endtime}" # Request format and time range
    f"&minmagnitude={minmag}" # Minimum magnitude filter
)

# Get data from the API
r = requests.get(url) # Send the HTTP GET request to the USGS API
data = r.json() # Parse the response as JSON format (common data format)
features = data['features'] # Extract the list of earthquake events

# Let's make a dataframe of earthquake events
# each event in the list obtained above contains features like location, magnitude, etc.
# we can extract them as follows:
df = pd.DataFrame({
    'longitude': [f['geometry']['coordinates'][0] for f in features],
    'latitude': [f['geometry']['coordinates'][1] for f in features],
    'magnitude': [f['properties']['mag'] for f in features],
    'depth': [f['geometry']['coordinates'][2] for f in features],
    'place': [f['properties']['place'] for f in features]
})

df

## **Question 1**

**Let's practice using requests! In the code cell below, query the USGS catalog for all major earthquakes (magnitude greater than 7) that occurred between January 1st 2011 and December 31st 2011. I've included some hints to outline your code below**

In [ ]:
# Define your parameters

# Set up your url (hint: we're querying the same API - so just copy from above!)

# Use requests to get data from the API

# Extract features of events into a dataframe (hint: look at above code)

# print dataframe

# II. Mapping in Python

In our last Python exercises, we explored one package for mapping in Python. Now, we're going to explore another package: the *Basemaps* package. It function is very similar, but it has some benefits (and tradeoffs). You may have noticed that when mapping topography in the last lab, some of the maps were *very* slow to load. As you'll see, *Basemaps* is very fast! That's because it gives us a pretty low resolution topography, which isn't great if we wanted to zoom in on a smaller area like NYC, but is perfectly fine for the global maps we're going to be making today!

Before we create our first map, we'll need some good data to work with. Thus, let's query for major earthquakes between 2000 and 2010. You've seen the code to do this before, so feel free to just run the code I've provided below. All I've changed from the code we worked with above is the start and end times!

In [ ]:
# You've seen all of this code before
# We're querying for 2000-2010
starttime = "2000-01-01"
endtime = "2010-12-31" 
minmag = 7 

url = (
    "https://earthquake.usgs.gov/fdsnws/event/1/query?" 
    f"format=geojson&starttime={starttime}&endtime={endtime}" 
    f"&minmagnitude={minmag}" 
)

r = requests.get(url) 
data = r.json() 
features = data['features'] 

df = pd.DataFrame({
    'longitude': [f['geometry']['coordinates'][0] for f in features],
    'latitude': [f['geometry']['coordinates'][1] for f in features],
    'magnitude': [f['properties']['mag'] for f in features],
    'depth': [f['geometry']['coordinates'][2] for f in features],
    'place': [f['properties']['place'] for f in features]
})

Now that we have our data, let's make our map! I've annotated the mapping code below to explain what each line does. Some important things to note about the map we're making:
1. We're using a cylindrical projection
2. We're coloring the data points based on the depth of the earthquake
3. We're defining the size of the data points based on the magnitude of the earthquake

In [ ]:
# Set up the figure and Basemap
fig = plt.figure(figsize=(12, 6)) # Create a figure of size 12x6 inches
m = Basemap(projection='cyl', resolution='l') # Use the cylindrical projection for the map

# Add topographic base
m.etopo(scale=0.5) 

# Draw elements
m.drawcoastlines() # Draw coastlines on the map
m.drawcountries() # Draw country borders
m.drawparallels(range(-90, 91, 30), labels=[1,0,0,0]) # Draw parallels (latitude lines)
m.drawmeridians(range(-180, 181, 60), labels=[0,0,0,1]) # Draw meridians (longitude lines)

# Convert lat/lon to map projection coordinates
x, y = m(df['longitude'].values, df['latitude'].values)

# Scatter the earthquake data
sc = m.scatter(x, y, c=df['depth'], cmap='jet', # color by depth
               s=df['magnitude']**2, edgecolor='k') # size by magnitude

# Colorbar and title
plt.colorbar(sc, label='Depth (km)')
plt.title("Major Earthquakes (2000 - 2010)")
plt.show()

Perhaps we were only interested in the earthquakes in South America. In our initial Basemap initialization, we can specify the specific latitude and longitude ranges we'd like to map:

In [ ]:
# Set up the figure and Basemap
fig = plt.figure(figsize=(12, 6)) # Create a figure of size 12x6 inches

###############
# The following line of code is the only one we've changed
###############

m = Basemap(projection='cyl', resolution='l',
            llcrnrlat=-60, urcrnrlat=15,   # Latitude bounds
            llcrnrlon=-100, urcrnrlon=-30)  # Longitude bounds (wraps the Pacific)

###############

# Add topographic base
m.etopo(scale=0.5) 

# Draw elements
m.drawcoastlines() # Draw coastlines on the map
m.drawcountries() # Draw country borders
m.drawparallels(range(-90, 91, 30), labels=[1,0,0,0]) # Draw parallels (latitude lines)
m.drawmeridians(range(-180, 181, 60), labels=[0,0,0,1]) # Draw meridians (longitude lines)

# Convert lat/lon to map projection coordinates
x, y = m(df['longitude'].values, df['latitude'].values)

# Scatter the earthquake data
sc = m.scatter(x, y, c=df['depth'], cmap='jet', # color by depth
               s=df['magnitude']**2, edgecolor='k') # size by magnitude

# Colorbar and title
plt.colorbar(sc, label='Depth (km)')
plt.title("Major Earthquakes (2000 - 2010)")
plt.show()

# III. Lab 08 - Part 2

In [ ]:
# QUESTION 2: Query for earthquakes (M > 5.5) between 2005 and 2025

In [ ]:
# QUESTION 2: Plot those earthquakes
# Be sure to: (1) color by DEPTH and (2) scale size by MAGNITUDE

In [ ]:
# QUESTION 6: Adjust lat and lon ranges of map!

In [ ]:
# QUESTION 8: Adjust lat and long ranges of map again!